In [1]:
import os
os.chdir("../")
%pwd

'c:\\Users\\Arnab 2001\\Desktop\\data_science_projects\\medical_chatbot\\End-to-End-Medical_chatbot'

In [2]:
from langchain.document_loaders import PyPDFLoader, DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [3]:
# Extracting text from the PDF files
def extract_text_from_pdfs(pdf_directory):
    loader = DirectoryLoader(pdf_directory, glob="*.pdf", loader_cls=PyPDFLoader, show_progress=True)
    documents = loader.load()
    return documents

In [4]:
extracted_documents = extract_text_from_pdfs("data")


100%|██████████| 1/1 [03:53<00:00, 233.23s/it]


In [5]:
print(f"Number of documents extracted: {len(extracted_documents)}")

Number of documents extracted: 637


In [6]:
from typing import List
from langchain.schema import Document

def filter_to_minimum_docs(docs: List[Document])-> List[Document]:
    minimum_docs: List[Document] = []
    for doc in docs:
        src = doc.metadata.get("source", "")
        minimum_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )
    return minimum_docs

In [7]:
minimum_docs = filter_to_minimum_docs(extracted_documents)
print(f"Number of minimum documents: {len(minimum_docs)}")

Number of minimum documents: 637


In [8]:
# Split the document into smaller chunks
def text_split(minimul_docs: List[Document]) -> List[Document]:
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=500, 
        chunk_overlap=20
    )
    text_chunks = text_splitter.split_documents(minimul_docs)
    return text_chunks

In [9]:
texts_chunk = text_split(minimum_docs)
print(f"Number of text chunks: {len(texts_chunk)}")

Number of text chunks: 5859


In [10]:
from langchain.embeddings import HuggingFaceEmbeddings
def create_embeddings(text_chunks: List[Document]):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    return embeddings

embeddings = create_embeddings(texts_chunk)


Caching the list of root modules, please wait!
(This will only be done once - type '%rehashx' to reset cache!)



C:\Users\Arnab 2001\AppData\Local\Temp\ipykernel_27140\1888492645.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
c:\Users\Arnab 2001\anaconda3\envs\medical_chatbot\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
vector = embeddings.embed_query("What are the symptoms of diabetes?")
print(vector[:5])

[0.026477515697479248, 0.01452952902764082, -0.04400854557752609, 0.09720121324062347, 0.004718074109405279]


In [13]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [14]:
GROQ_API_KEY=os.getenv("GROQ_API_KEY")
PINECONE_API_KEY=os.getenv("PINECONE_API_KEY")

In [15]:
from pinecone import Pinecone
pc = Pinecone(api_key=PINECONE_API_KEY)
pc

In [16]:
from pinecone import ServerlessSpec
index_name = "medical-chatbot"
if not pc.has_index(index_name):
    pc.create_index(
        name=index_name,
        dimension=384,  # Dimension of the embeddings
        metric="cosine",  # Similarity metric
        serverless=ServerlessSpec(cloud="aws", region="us-east-1")  # Serverless configuration
    )

index = pc.Index(index_name)
print(f"Index '{index_name}' is ready for use.")

Index 'medical-chatbot' is ready for use.


In [18]:
from langchain_pinecone import PineconeVectorStore
#docsearch = PineconeVectorStore.from_documents(
#   documents=texts_chunk,
#    embedding=embeddings,
#    index_name=index_name
#)

In [19]:
docsearch = PineconeVectorStore.from_existing_index(
    index_name=index_name,
    embedding=embeddings
)

In [20]:
retriever = docsearch.as_retriever(search_type="similarity", search_kwargs={"k":3})

In [21]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='f5bb315e-c09a-48aa-89db-29bbab72c9fa', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='cacf77d0-f494-412d-80dd-81093c1a48ed', metadata={'source': 'data\\Medical_book.pdf'}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='e98ae757-ad18-43e1-937f-ab1a3a39f43f', metadata={'source': 'data\\Medical_book.pdf'}, page_content='Acidosis see Respiratory acidosis; Renal\ntubular acidosis; Metabolic acidosis\nAcne\nDefinition\nAcne is a common skin disease characterized by\npimples on the face, chest, and back. It occurs when the\npores of the skin become clogged wi

In [40]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    model_name="llama-3.1-8b-instant",
    temperature=0.5
)

In [41]:
query = "What are symptoms of diabetes?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs):
    print(f"\n📄 Document {i+1}:\n")
    print(doc.page_content)


📄 Document 1:

• Type I diabetes mellitus. Characterized by fatigue and
an abnormally high level of glucose in the blood
(hyperglycemia).
• Amyotrophic lateral schlerosis. First signs are stum-
bling and difficulty climbing stairs. Later, muscle
cramps and twitching may be observed as well as
weakness in the hands making fastening buttons or
turning a key difficult. Speech may become slowed or
slurred. There may also be difficluty swallowing. As
respiratory muscles atrophy, there is increased danger

📄 Document 2:

begin to fall. A person with diabetes mellitus either does
not make enough insulin, or makes insulin that does not
work properly. The result is blood sugar that remains
high, a condition called hyperglycemia.
Diabetes must be diagnosed as early as possible. If
left untreated, it can damage or cause failure of the eyes,
kidneys, nerves, heart, blood vessels, and other body
organs. Hypoglycemia, or low blood sugar, may also be
discovered through blood sugar testing. Hypoglyce

In [42]:
from langchain.prompts import PromptTemplate

prompt = PromptTemplate(
    template="""
You are a medical assistant for question-answering tasks.
Use the following pieces of retrieved context to answer the question.
If you don't know the answer, say that you don't know.
Use three sentences maximum and keep the answer concise.

Context:
{context}

Question:
{input}

Answer:
""",
    input_variables=["context", "input"]
)

In [43]:
from langchain.chains.combine_documents import create_stuff_documents_chain

document_chain = create_stuff_documents_chain(
    llm,
    prompt
)

In [44]:
from langchain.chains import create_retrieval_chain

retrieval_chain = create_retrieval_chain(
    retriever,
    document_chain
)

In [45]:
response = retrieval_chain.invoke({
    "input": "What causes diabetes?"
})

print(response["answer"])

A person with diabetes mellitus either does not make enough insulin, or makes insulin that does not work properly, resulting in high blood sugar levels. This can be due to the pancreas not producing enough insulin or the body's inability to use the insulin produced.
